# Accuracy by disease gene count

This notebook reads the accuracy outputs from `accuracy/<number>_gene_related_disease/`.

It keeps every disease as one row and plots:

- x-axis: number of genes related to disease, taken from folder names like `44_gene_related_disease`
- y-axis: model accuracy, using `accuracy_any_causal_gene_percent`

Outputs are saved under `accuracy/`.

In [ ]:
from pathlib import Path
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("/Users/miasmacbook/Desktop/kcl/6-months_project")
ACCURACY_ROOT = PROJECT_ROOT / "accuracy"

PER_DISEASE_OUTPUT_CSV = ACCURACY_ROOT / "accuracy_by_gene_count_each_disease.csv"
GROUPED_OUTPUT_CSV = ACCURACY_ROOT / "accuracy_by_gene_count_grouped.csv"
OUTPUT_PNG = ACCURACY_ROOT / "accuracy_by_gene_count.png"

print(f"Accuracy root: {ACCURACY_ROOT}")

In [ ]:
summary_files = sorted(ACCURACY_ROOT.glob("*_gene_related_disease/final_gene_set_accuracy_summary.csv"))
print(f"Accuracy summary files found: {len(summary_files)}")

rows = []
for path in summary_files:
    df = pd.read_csv(path)
    if df.empty:
        continue

    folder_name = path.parent.name
    match = re.match(r"(\d+)_gene_related_disease", folder_name)
    if not match:
        continue

    n_genes = int(match.group(1))
    for _, row in df.iterrows():
        rows.append(
            {
                "disease_id": row["disease_id"],
                "folder_name": folder_name,
                "n_genes": n_genes,
                "n_repeats": row["n_repeats"],
                "n_causal_genes": row["n_causal_genes"],
                "accuracy_percent": row["accuracy_any_causal_gene_percent"],
                "repeat_count_with_any_causal_gene": row["repeat_count_with_any_causal_gene"],
                "similarity_hit_percent": row["similarity_hit_percent"],
                "mean_causal_gene_recovery_percent": row["mean_causal_gene_recovery_percent"],
                "summary_file": path.as_posix(),
            }
        )

results_df = pd.DataFrame(rows)
if results_df.empty:
    raise ValueError(f"No accuracy summary rows found under {ACCURACY_ROOT}")

results_df = results_df.sort_values(["n_genes", "disease_id"]).reset_index(drop=True)
results_df.to_csv(PER_DISEASE_OUTPUT_CSV, index=False)
print(f"Saved per-disease CSV: {PER_DISEASE_OUTPUT_CSV}")
results_df.head()

In [ ]:
group_df = (
    results_df.groupby("n_genes", as_index=False)
    .agg(
        mean_accuracy_percent=("accuracy_percent", "mean"),
        median_accuracy_percent=("accuracy_percent", "median"),
        min_accuracy_percent=("accuracy_percent", "min"),
        max_accuracy_percent=("accuracy_percent", "max"),
        mean_similarity_hit_percent=("similarity_hit_percent", "mean"),
        mean_causal_gene_recovery_percent=("mean_causal_gene_recovery_percent", "mean"),
        n_diseases=("disease_id", "nunique"),
    )
    .sort_values("n_genes")
    .reset_index(drop=True)
)

group_df.to_csv(GROUPED_OUTPUT_CSV, index=False)
print(f"Saved grouped CSV: {GROUPED_OUTPUT_CSV}")
group_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.scatter(
    results_df["n_genes"],
    results_df["accuracy_percent"],
    alpha=0.35,
    color="#2563eb",
    label="Each disease",
)

ax.plot(
    group_df["n_genes"],
    group_df["mean_accuracy_percent"],
    marker="o",
    linewidth=2.5,
    color="#dc2626",
    label="Mean accuracy",
)

ax.set_xlabel("Number of genes related to disease", fontsize=12)
ax.set_ylabel("Accuracy: final set contains causal gene (%)", fontsize=12)
ax.set_title("Model accuracy by disease gene count", fontsize=15, pad=12)
ax.set_ylim(0, 105)
ax.grid(True, axis="y", color="#cbd5e1", linewidth=0.8, alpha=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="best")

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150)
plt.close(fig)

print(f"Saved graph: {OUTPUT_PNG}")